In [1]:
%run yagp.ipynb

In [2]:
import regex as re
import pandas as pd

# tokens that are common in real names in this file
allowed_particles = {
    "DE", "DEL", "DELA", "DI", "DA", "DOS", "DAS", "DU",
    "LA", "LE", "VAN", "VON", "ST", "ST.", "O"
}

# tokens that appear in dance/piece/company lines in this file
bad_tokens = {
    "ACADEMY", "SCHOOL", "STUDIO", "STUDIOS", "CONSERVATORY", "COMPANY",
    "THEATER", "THEATRE", "CENTER", "CENTRE", "ARTS", "DANCENTER",
    "PAS", "DEUX", "TROIS", "VARIATION", "VARIATIONS", "DANCE", "SCENE",
    "ACT", "SUITE", "WALTZ", "PRELUDE", "REQUIEM", "FESTIVAL", "NUTCRACKER",
    "GISELLE", "COPPELIA", "RAYMONDA", "HARLEQUINADE", "SATANELLA",
    "SYLPHIDE", "CARMEN", "FLOWER", "FLOWERS", "DREAM", "BALL", "PEARLS",
    "SWAN", "LAKE", "CORSAIRE", "PAQUITA", "ESMERELDA", "WALPURGIS",
    "TALISMAN", "BAYADERE", "FUGUE", "DOLLS", "VOICE", "VISION", "STORM",
    "WORLD", "ILLUSION", "CAPTIVE", "INTRIGUE", "REMINISCENCE",
    "AND", "FROM", "THE", "OF", "IN", "TO", "WITH", "OVER", "UNDER",
    "AFTER", "FOR", "MY", "ME", "YOU", "YOUR", "OUR", "HIS", "HER",
    "THEM", "HEAR", "EXCERPT", "MOVEMENT", "PART", "BIG", "SMALL"
}

# line should still have name-like structure
name_shape = re.compile(
    r"""^[\p{Lu}][\p{L}'´.`\-]*
        (?:\s+[\p{Lu}][\p{L}'´.`\-]*){1,5}
        (?:,\s*[A-Z]{2})?$""",
    re.VERBOSE,
)

def is_person_name(line: str) -> bool:
    if pd.isna(line):
        return False

    line = str(line).strip()
    if not line:
        return False

    # basic structure
    if not name_shape.fullmatch(line):
        return False

    # reject obvious punctuation/title patterns
    if re.search(r'\(\d+\)|[#":;/?]', line):
        return False

    # split off optional trailing state
    core = re.sub(r",\s*[A-Z]{2}$", "", line)

    tokens = core.split()

    # reject if any token is a known title/company token,
    # unless it is an allowed name particle
    for tok in tokens:
        cleaned = tok.strip(".'’´`-").upper()
        if cleaned in bad_tokens and cleaned not in allowed_particles:
            return False

    return True


with open("yagp_names.txt", "r", encoding="utf-8") as f:
    lines = [line.strip() for line in f]

names_only = [line for line in lines if is_person_name(line)]

with open("filtered_person_names.txt", "w", encoding="utf-8") as f:
    for name in names_only:
        f.write(name + "\n")

print(names_only[:50])
print(f"Total names kept: {len(names_only)}")

['KORY GLATMAN, NJ', 'ASHTON BERKLEY', 'DANA GAULE', 'KATRINA DALTON', 'TIFFANY SMITH', 'MELANIE HAMRICK', 'SHANE WUERTHNER', 'KARA COOPER', 'ASHLEY CANTERNA', 'DARLI NOGARR', 'DANIELLE OLDZIEY', 'ANGELA RAMAKIS', 'BARRY KAROLLIS', 'SARAH MAIRA', 'ANDREA EMMONS', 'MATTHEW DONNELL', 'LINSEY LARSEN', 'CHRISTIAN TWORZYANSKI', 'VICTOR KASATSKY', 'AYAKO TAKAHASHI', 'ALISON DEANE', 'AIMI TOYAMA', 'KIYOSHI KAWANO', 'AMY BRIONES', 'LAURA TSIM', 'LAUREN ASHLEY', 'FABI CALBORN', 'STACEY AUNG', 'MEGAN VAN WINKLE', 'KATIE SHERMAN', 'ELIZABETH LAPENA', 'CHLOE FELESINA', 'MICHELLE CARPENTER', 'CARRIE JUDSON', 'ALEXI AGDEPPA', 'JULIA OLSEN-RODRIGUEZ', 'JENNIFER WHALEN', 'NICK SCOTT', 'GINGER SMITH', 'SHERRY MORAY', 'NICOLE STRIMELL', 'MARCIE MANGAN', 'TARASINA MASI', 'NICOLE MANCUSO', 'JESSICA DE JESUS', 'AMANDA ULIBARRI', 'MARINA IKEMOTO', 'ASHLEY DAHM', 'ANGELINA ZUCCARINI', 'HEATHER CHIN']
Total names kept: 14633


In [3]:
false_positives = [line for line in names_only if not is_person_name(line)]
print(false_positives[:100])

[]


In [4]:
suspects = [
    line for line in names_only
    if any(tok in bad_tokens for tok in re.sub(r",\s*[A-Z]{2}$", "", line).split())
]
print(suspects[:100])

[]
